# Scaling Ambiguity Augmenting Human Annotation in Speech Emotion Recognition with Audio-Language Models - Ambiguity Level Analysis

Generates Figure 3: **JS Divergence by ambiguity level** for **IEMOCAP** and **MSP-Podcast**, comparing **human-only** vs. **combined** annotation training.
 
Ambiguity is defined by the normalized **Shannon Entropy** of the ground-truth emotion list per utterance. Samples are split into tertiles (low / medium / high) based on entropy rank, matching the evaluation protocol in **Section 3.3** of the paper.

## Outputs
-------
- fig3a.pdf : JS Divergence vs. ambiguity level (Low / Medium / High) for IEMOCAP
- fig3b.pdf : JS Divergence vs. ambiguity level (Low / Medium / High) for MSP-Podcast

## Dependencies Import

In [ ]:
import sys
sys.path.append('..') 

import json
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from scipy.spatial.distance import jensenshannon

from lib import load_data as ld

## Section 1: Data Loading

In [ ]:
# Human-only test annotations (list format: {"emotion": ["happy", "sad", ...]})
test_data_iemo = ld.load_json("../processed_data/iemocap_test_raw_annotations.json")
print(f"Loaded {len(test_data_iemo)} IEMOCAP human test samples")
 
test_data_msp = ld.load_json("../processed_data/msp_test_raw_annotations.json")
print(f"Loaded {len(test_data_msp)} MSP-Podcast human test samples")
 
# Combined (human + synthetic) test annotations — used as ground truth for
# the ensemble model evaluation
test_data_combined_iemo = ld.load_json("../processed_data/combined_annotations/combined_iemocap_test_raw_annotations.json")
print(f"Loaded {len(test_data_combined_iemo)} IEMOCAP combined test samples")
 
test_data_combined_msp = ld.load_json("../processed_data/combined_annotations/combined_msp_test_raw_annotations.json")
print(f"Loaded {len(test_data_combined_msp)} MSP-Podcast combined test samples")

## Section 2: Ambiguity Grouping Utilities

In [ ]:
def calculate_entropy(emotions):
    
    """
    Compute normalized Shannon entropy over a list of emotion labels.
 
    Entropy is normalized by the maximum possible entropy for the number of
    unique emotions present in this sample, so the output is in [0, 1].
    A value of 0 means unanimous agreement; 1 means maximum disagreement.
 
    Parameters
    ----------
    emotions : list of str
        Raw annotation labels for a single utterance.
 
    Returns
    -------
    float
        Normalized entropy in [0, 1]. Returns 0 for empty input.
    """
    
    if not emotions:
        return 0
    
    # Count the number of occurrences of each emotion
    emotion_counts = Counter(emotions)
    total_annotators = len(emotions)
    
    # Calculate entropy
    entropy = 0
    for count in emotion_counts.values():
        prob = count / total_annotators
        if prob > 0:  # Avoid log(0)
            entropy -= prob * np.log(prob)
    
    # Get the number of unique emotions in this sample
    n_unique_emotions = len(emotion_counts)
    
    # Normalize entropy by maximum possible entropy for this number of categories
    max_entropy = np.log(n_unique_emotions) if n_unique_emotions > 1 else 1
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    
    return normalized_entropy

def get_ambiguity_groups_by_rank(test_data):
    
    """
    Divide utterances into low / medium / high ambiguity tertiles based on
    entropy rank, matching the evaluation protocol in Section 3.3.
 
    Parameters
    ----------
    test_data : list of dict
        Each item must have "id" and "emotion" (list format) fields.
 
    Returns
    -------
    ambiguity_groups : dict
        {"low": [id, ...], "medium": [id, ...], "high": [id, ...]}
    low_threshold : float
        Entropy value at the 33rd percentile boundary.
    high_threshold : float
        Entropy value at the 67th percentile boundary.
    """
    
    samples_with_entropy = []
    
    for sample in test_data:
        if 'emotion' in sample and isinstance(sample['emotion'], list) and len(sample['emotion']) > 0:
            entropy = calculate_entropy(sample['emotion'])
            samples_with_entropy.append((sample['id'], entropy))
    
    samples_with_entropy.sort(key=lambda x: x[1])
    
    n = len(samples_with_entropy)
    third = n // 3
    
    low_threshold = samples_with_entropy[third-1][1] if third > 0 else 0
    high_threshold = samples_with_entropy[2*third-1][1] if 2*third < n else samples_with_entropy[-1][1]
    
    ambiguity_groups = {
        'low': [sample_id for sample_id, _ in samples_with_entropy[:third]],
        'medium': [sample_id for sample_id, _ in samples_with_entropy[third:2*third]],
        'high': [sample_id for sample_id, _ in samples_with_entropy[2*third:]]
    }
    
    return ambiguity_groups, low_threshold, high_threshold

## Section 3: Performance Evaluation by Ambiguity Level

In [ ]:
def calculate_model_performance_by_ambiguity(predictions, ground_truths, ambiguity_groups, emotions):
    """
    Compute accuracy and mean JS divergence for each ambiguity group.
 
    Ground truths are provided in raw list format and converted to probability
    distributions internally. Predictions are expected in distribution format
    (dict of {emotion: probability}).
 
    Parameters
    ----------
    predictions : list of dict
        Model output, each with "id" and "emotion" (distribution format).
    ground_truths : list of dict
        Reference annotations, each with "id" and "emotion" (list format).
    ambiguity_groups : dict
        Output of get_ambiguity_groups_by_rank().
    emotions : list of str
        Ordered list of emotion categories for vector alignment.
 
    Returns
    -------
    dict
        {ambiguity_level: {"accuracy": float, "js_divergence": float,
                           "sample_count": int}}
    """
    pred_dict  = {item['id']: item for item in predictions}
    truth_dict = {item['id']: item for item in ground_truths}
 
    performance_by_ambiguity = {}
 
    for ambiguity, sample_ids in ambiguity_groups.items():
        matched_preds  = []
        matched_truths = []
 
        for sample_id in sample_ids:
            if sample_id not in pred_dict or sample_id not in truth_dict:
                continue
 
            pred_emotion  = pred_dict[sample_id]['emotion']   # distribution dict
            truth_emotion = truth_dict[sample_id]['emotion']  # annotation list
 
            # Convert list format to probability distribution
            if isinstance(truth_emotion, list) and len(truth_emotion) > 0:
                emotion_counts = Counter(truth_emotion)
                total = len(truth_emotion)
                truth_dist = {k: v / total for k, v in emotion_counts.items()}
 
                if pred_emotion and truth_dist:
                    matched_preds.append(pred_emotion)
                    matched_truths.append(truth_dist)
 
        if not matched_preds:
            performance_by_ambiguity[ambiguity] = {
                'accuracy': 0, 'js_divergence': 0, 'sample_count': 0
            }
            continue
 
        total = len(matched_preds)
 
        # Accuracy: dominant emotion of predicted vs. ground-truth distribution
        correct = sum(
            1 for pred, truth in zip(matched_preds, matched_truths)
            if pred and truth
            and max(pred, key=pred.get) == max(truth, key=truth.get)
        )
        accuracy = correct / total
 
        # Mean JS divergence over the ambiguity group
        js_values = []
        for pred, truth in zip(matched_preds, matched_truths):
            pred_vec  = np.array([pred.get(e, 0) for e in emotions])
            truth_vec = np.array([truth.get(e, 0) for e in emotions])
 
            # Normalise to ensure valid probability vectors
            if np.sum(pred_vec) > 0:
                pred_vec /= np.sum(pred_vec)
            if np.sum(truth_vec) > 0:
                truth_vec /= np.sum(truth_vec)
 
            js_values.append(jensenshannon(pred_vec, truth_vec))
 
        performance_by_ambiguity[ambiguity] = {
            'accuracy':     accuracy,
            'js_divergence': np.mean(js_values),
            'sample_count': total
        }
 
    return performance_by_ambiguity

## Section 4: Figure Generation

### Figure 3a: IEMOCAP

In [ ]:
ambiguity_groups_human, low_threshold, high_threshold = get_ambiguity_groups_by_rank(test_data_iemo)

print(f"Test set entropy thresholds from 1/3 division:")
print(f"  Low threshold (33.33% percentile): {low_threshold:.3f}")
print(f"  High threshold (66.67% percentile): {high_threshold:.3f}")

base_predictions = ld.load_json("../model_finetuning/evaluation_results_iemocap/SMOTE_30p/iemocap_predictions_tNone_aNone.json")
ensemble_predictions = ld.load_json("../model_finetuning/evaluation_results_combined-iemocap/Combined_25p/combined-iemocap_predictions_tNone_aNone.json")

print("Sample distribution by ambiguity level:")
for ambiguity, samples in ambiguity_groups_human.items():
    print(f"  {ambiguity.capitalize()} ambiguity: {len(samples)} samples")

# Define emotions based on the dataset
if 'Sadness' in test_data_iemo[0]['emotion']:
    emotions = ["Anger", "Happiness", "Neutral state", "Sadness"]
else:
    emotions = ["Angry", "Happy", "Neutral", "Sad"]

# Calculate model performance by ambiguity level 
base_performance = calculate_model_performance_by_ambiguity(
    base_predictions, test_data_iemo, ambiguity_groups_human, emotions)
ensemble_performance = calculate_model_performance_by_ambiguity(
    ensemble_predictions, test_data_combined_iemo, ambiguity_groups_human, emotions)

# Plotting the JS Divergence by sample ambiguity
fig, ax = plt.subplots(1, 1, figsize=(4, 3))

# Define ambiguity levels and labels
ambiguities = ['low', 'medium', 'high']
ambiguity_labels = [
    'Low Ambiguity\n(Bottom 1/3)', 
    'Medium Ambiguity\n(Middle 1/3)', 
    'High Ambiguity\n(Top 1/3)'
]
models = ['Human', 'Combined']

# Width of each bar
bar_width = 0.65
metric_group_spacing = 2

# Calculate positions for each metric group
positions = np.arange(len(ambiguities)) * metric_group_spacing

# Set positions for bars
r1 = positions
r2 = positions + bar_width*1.3

# Colors for the models
colors = ['royalblue', '#E74C3C']
edge_colors = ['navy', '#C0392B']
best_edge_color = '#9E0B0B'

# Extract JS Divergence values for each ambiguity level
base_values = [base_performance[d]['js_divergence'] for d in ambiguities]
ensemble_values = [ensemble_performance[d]['js_divergence'] for d in ambiguities]

# Create bars for each model
bars1 = ax.bar(r1, base_values, bar_width, label=models[0], 
                color=colors[0], alpha=0.7, edgecolor=edge_colors[0], linewidth=1.3)
bars2 = ax.bar(r2, ensemble_values, bar_width, label=models[1], 
                color=colors[1], alpha=0.7, edgecolor=edge_colors[1], linewidth=1.3)

all_bars = [bars1, bars2]
all_values = [base_values, ensemble_values]

# Add labels and highlight the best performance
for j, ambiguity in enumerate(ambiguities):
    # Add labels to each bar
    for k, (bars, values) in enumerate(zip(all_bars, all_values)):
        bar = bars[j]
        value = values[j]
        
        # Format the value text
        val_text = f'{value:.3f}'
        
        # Annotate the bar with the value
        height = bar.get_height()
        ax.annotate(val_text, (bar.get_x() + bar.get_width()/2., height), 
                    xytext=(0, 12), textcoords="offset points",
                    ha='center', va='center', fontsize=10, fontweight='bold')

# Set labels and title
ax.set_ylabel('JS Divergence', fontweight='bold', fontsize=13)
ax.set_xlabel('Ambiguity Level', fontweight='bold', fontsize=13)

ambiguity_labels = [
    'Low',
    'Medium',
    'High'
]
group_centers = positions + bar_width*0.65
ax.set_xticks(group_centers)
ax.set_xticklabels(ambiguity_labels, fontsize=13)

# Set Y-axis limits
all_vals = base_values + ensemble_values
y_max = max(all_vals) * 1.26
ax.set_ylim(0, y_max)
ax.set_yticks(np.arange(0, y_max, step=0.1))
ax.tick_params(axis='y', labelsize=13)

# Add grid and background styling
ax.grid(True, linestyle='-', alpha=0.3, zorder=0, color='gray')
ax.set_facecolor('white')

# Adjust layout and save the figure
plt.tight_layout()
plt.savefig('fig3a.pdf', format='pdf', dpi=600, 
            bbox_inches='tight', facecolor='white')
plt.show()

# Print numerical summary
print("\n" + "="*80)
print("PERFORMANCE BY SAMPLE AMBIGUITY LEVEL - NUMERICAL SUMMARY")
print("="*80)

for ambiguity in ambiguities:
    print(f"\n{ambiguity.upper()} AMBIGUITY SAMPLES:")
    print(f"  Sample count: {len(ambiguity_groups_human[ambiguity])}")
    print(f"  Base Model     - Accuracy: {base_performance[ambiguity]['accuracy']:.3f}, JS: {base_performance[ambiguity]['js_divergence']:.3f}")
    print(f"  Ensemble       - Accuracy: {ensemble_performance[ambiguity]['accuracy']:.3f}, JS: {ensemble_performance[ambiguity]['js_divergence']:.3f}")

### Figure 3b: MSP-Podcast

In [ ]:
ambiguity_groups_human, low_threshold, high_threshold = get_ambiguity_groups_by_rank(test_data_msp)

print(f"Test set entropy thresholds from 1/3 division:")
print(f"  Low threshold (33.33% percentile): {low_threshold:.3f}")
print(f"  High threshold (66.67% percentile): {high_threshold:.3f}")

base_predictions = ld.load_json("../model_finetuning/evaluation_results_msp/human_SMOTE_25p/msp_predictions_tNone_aNone.json")
ensemble_predictions = ld.load_json("../model_finetuning/evaluation_results_combined-msp/Combined_20p/combined-msp_predictions_tNone_aNone.json")


print("Sample distribution by ambiguity level:")
for ambiguity, samples in ambiguity_groups_human.items():
    print(f"  {ambiguity.capitalize()} ambiguity: {len(samples)} samples")

# Define emotions based on the dataset
if 'Sadness' in test_data_msp[0]['emotion']:
    emotions = ["Anger", "Happiness", "Neutral state", "Sadness"]
else:
    emotions = ["Angry", "Happy", "Neutral", "Sad"]

# Calculate model performance by ambiguity level 
base_performance = calculate_model_performance_by_ambiguity(
    base_predictions, test_data_msp, ambiguity_groups_human, emotions)
ensemble_performance = calculate_model_performance_by_ambiguity(
    ensemble_predictions, test_data_combined_msp, ambiguity_groups_human, emotions)

# Plotting the JS Divergence by sample ambiguity
fig, ax = plt.subplots(1, 1, figsize=(4, 3))

# Define ambiguity levels and labels
ambiguities = ['low', 'medium', 'high']
ambiguity_labels = [
    'Low Ambiguity\n(Bottom 1/3)', 
    'Medium Ambiguity\n(Middle 1/3)', 
    'High Ambiguity\n(Top 1/3)'
]
models = ['Human', 'Combined']

# Width of each bar
bar_width = 0.65
metric_group_spacing = 2

# Calculate positions for each metric group
positions = np.arange(len(ambiguities)) * metric_group_spacing

# Set positions for bars
r1 = positions
r2 = positions + bar_width*1.3

# Colors for the models
colors = ['royalblue', '#E74C3C']
edge_colors = ['navy', '#C0392B']
best_edge_color = '#9E0B0B'

# Extract JS Divergence values for each ambiguity level
base_values = [base_performance[d]['js_divergence'] for d in ambiguities]
ensemble_values = [ensemble_performance[d]['js_divergence'] for d in ambiguities]

# Create bars for each model
bars1 = ax.bar(r1, base_values, bar_width, label=models[0], 
                color=colors[0], alpha=0.7, edgecolor=edge_colors[0], linewidth=1.3)
bars2 = ax.bar(r2, ensemble_values, bar_width, label=models[1], 
                color=colors[1], alpha=0.7, edgecolor=edge_colors[1], linewidth=1.3)

all_bars = [bars1, bars2]
all_values = [base_values, ensemble_values]

# Add labels and highlight the best performance
for j, ambiguity in enumerate(ambiguities):
    # Add labels to each bar
    for k, (bars, values) in enumerate(zip(all_bars, all_values)):
        bar = bars[j]
        value = values[j]
        
        # Format the value text
        val_text = f'{value:.3f}'
        
        # Annotate the bar with the value
        height = bar.get_height()
        ax.annotate(val_text, (bar.get_x() + bar.get_width()/2., height), 
                    xytext=(0, 12), textcoords="offset points",
                    ha='center', va='center', fontsize=10, fontweight='bold')

# Set labels and title
ax.set_xlabel('Ambiguity Level', fontweight='bold', fontsize=13)

ambiguity_labels = [
    'Low',
    'Medium',
    'High'
]
group_centers = positions + bar_width*0.65
ax.set_xticks(group_centers)
ax.set_xticklabels(ambiguity_labels, fontsize=13)

# Set Y-axis limits
all_vals = base_values + ensemble_values
y_max = max(all_vals) * 1.43
ax.set_ylim(0, y_max)
ax.set_yticks(np.arange(0, y_max, step=0.1))
ax.tick_params(axis='y', labelsize=13)

# Add grid and background styling
ax.grid(True, linestyle='-', alpha=0.3, zorder=0, color='gray')
ax.set_facecolor('white')

# Add legend
ax.legend(loc='upper center', ncol=2, prop={'size': 11, 'weight': 'bold'})

# Adjust layout and save the figure
plt.tight_layout()
plt.savefig('fig3b.pdf', format='pdf', dpi=600, 
            bbox_inches='tight', facecolor='white')
plt.show()

# Print numerical summary
print("\n" + "="*80)
print("PERFORMANCE BY SAMPLE AMBIGUITY LEVEL - NUMERICAL SUMMARY")
print("="*80)

for ambiguity in ambiguities:
    print(f"\n{ambiguity.upper()} AMBIGUITY SAMPLES:")
    print(f"  Sample count: {len(ambiguity_groups_human[ambiguity])}")
    print(f"  Base Model     - Accuracy: {base_performance[ambiguity]['accuracy']:.3f}, JS: {base_performance[ambiguity]['js_divergence']:.3f}")
    print(f"  Ensemble       - Accuracy: {ensemble_performance[ambiguity]['accuracy']:.3f}, JS: {ensemble_performance[ambiguity]['js_divergence']:.3f}")